In [1]:

# Cell 1 â€” Kaggle-specific Installation
!pip install pip3-autoremove -q
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121 -q
!pip install unsloth -q
!pip install transformers datasets trl peft accelerate bitsandbytes -q
!pip install wandb evaluate rouge_score nltk -q

# NEW FIX: Remove torchao to prevent the int1 import crash
!pip uninstall torchao -y

torchaudio 2.10.0+cu128 (/usr/local/lib/python3.12/dist-packages)
    torch 2.10.0+cu128 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cublas-cu12 12.8.4.1 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cuda-cupti-cu12 12.8.90 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cuda-nvrtc-cu12 12.8.93 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cuda-runtime-cu12 12.8.90 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cudnn-cu12 9.10.2.21 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cufft-cu12 11.3.3.83 (/usr/local/lib/python3.12/dist-packages)
            nvidia-nvjitlink-cu12 12.8.93 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cufile-cu12 1.13.1.3 (/usr/local/lib/python3.12/dist-packages)
        nvidia-curand-cu12 10.3.9.90 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cusolver-cu12 11.7.3.90 (/usr/local/lib/python3.12/dist-packages)
            nvidia-cusparse-cu12 12.5.8.93 (/usr/local/lib/python3.1

In [1]:
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 15.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 32.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 33.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.0 MB/s eta 0:0

In [2]:
# Cell 2 â€” Kaggle Secrets for Login
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

user_secrets = UserSecretsClient()

# Pull keys securely from Kaggle Add-ons > Secrets
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
hf_key = user_secrets.get_secret("HF_TOKEN")

wandb.login(key=wandb_key)
login(token=hf_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kumarsarthakofficial (kumarsarthakofficial-birla-institute-of-technology-mesra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:

# Cell 3 — Load Llama 3.2-3B with stronger LoRA config
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

# KEY UPGRADE: r=64 (was 16) — much more expressive for domain tasks
model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # Higher rank = more capacity to learn domain knowledge
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,          # alpha = 2x rank is a common best practice
    lora_dropout=0.05,       # Small dropout for regularization
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.15 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [4]:
# Cell 4 - Load and prepare medical Q&A datasets
from datasets import load_dataset, concatenate_datasets
import json

# Load a small medical multiple-choice dataset for quick experiments
medquad = load_dataset("lavita/medical-qa-shared-task-v1-toy", split="train")

# Load PubMedQA - biomedical research Q&A
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

print(f"Medical toy set: {len(medquad)} examples")
print(f"PubMedQA: {len(pubmedqa)} examples")

# Inspect the structure
print("\nMedical toy sample:", medquad[0])
print("\nPubMedQA sample:", pubmedqa[0])

README.md:   0%|          | 0.00/773 [00:00<?, ?B/s]

data/train-00000-of-00001-dbf9914f90b9c0(…):   0%|          | 0.00/44.3k [00:00<?, ?B/s]

data/dev-00000-of-00001-c4e476938006b653(…):   0%|          | 0.00/45.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/32 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Medical toy set: 32 examples
PubMedQA: 1000 examples

Medical toy sample: {'id': 2622, 'ending0': 'Pallor, cyanosis, and erythema of the hands', 'ending1': 'Calcium deposits on digits', 'ending2': 'Blanching vascular abnormalities', 'ending3': 'Hypercoagulable state', 'ending4': 'Heartburn and regurgitation', 'label': 3, 'sent1': 'A 35-year-old woman comes to your office with a variety of complaints.  As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. ', 'sent2': 'All of the following symptoms and signs would be expected to be present EXCEPT:', 'startphrase': 'A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:'}

PubMedQA sample: {'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling l

In [5]:
# Cell 5 - Format into ChatML

def format_medquad(example):
    """Format the toy medical multiple-choice dataset into ChatML."""
    question = example.get("startphrase") or f"{example.get('sent1', '').strip()} {example.get('sent2', '').strip()}".strip()
    options = [example[f"ending{i}"] for i in range(5) if example.get(f"ending{i}")]
    option_text = "\n".join(f"{idx + 1}. {choice}" for idx, choice in enumerate(options))

    answer_index = int(example["label"])
    answer = options[answer_index] if 0 <= answer_index < len(options) else ""

    return {
        "messages": [
            {
                "role": "system",
                "content": "You are a knowledgeable medical AI assistant. Read the medical question carefully and return the best supported answer."
            },
            {
                "role": "user",
                "content": f"{question}\n\nOptions:\n{option_text}"
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]
    }


def format_pubmedqa(example):
    """Format PubMedQA - it has context, question, and long_answer."""
    context = " ".join(example["context"]["contexts"][:2])
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are a knowledgeable medical AI assistant specializing in biomedical research. Provide accurate, evidence-based answers."
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {example['question']}"
            },
            {
                "role": "assistant",
                "content": example["long_answer"]
            }
        ]
    }

# Process datasets
medquad_formatted = medquad.map(format_medquad, remove_columns=medquad.column_names)
pubmedqa_formatted = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa.column_names)

# Keep the tiny medical toy set and cap PubMedQA for a fast single-GPU run
medquad_subset = medquad_formatted.select(range(min(15000, len(medquad_formatted))))
pubmedqa_subset = pubmedqa_formatted.select(range(min(5000, len(pubmedqa_formatted))))

# Combine them
sft_dataset = concatenate_datasets([medquad_subset, pubmedqa_subset]).shuffle(seed=42)
print(f"\nFinal SFT dataset: {len(sft_dataset)} examples")
print("\nSample formatted example:")
print(json.dumps(sft_dataset[0], indent=2))

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Final SFT dataset: 1032 examples

Sample formatted example:
{
  "messages": [
    {
      "content": "You are a knowledgeable medical AI assistant specializing in biomedical research. Provide accurate, evidence-based answers.",
      "role": "system"
    },
    {
      "content": "Context: This study examines whether having a regular clinician for preventive care is associated with quality of care for young children, as measured by interpersonal quality ratings and content of anticipatory guidance. The National Survey of Early Childhood Health (NSECH), a nationally representative parent survey of health care quality for 2068 young US children fielded by the National Center for Health Statistics (NCHS).\n\nQuestion: Does having a regular primary care clinician improve quality of preventive care for young children?",
      "role": "user"
    },
    {
      "content": "Having a regular primary care clinician is embraced in pediatrics, although team care among physicians is also widely pr

In [8]:
# Cell 6 â€” Apply chat template (this is how production models format data)
def apply_chat_template(example):
    """Apply the model's built-in chat template to messages"""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

sft_dataset = sft_dataset.map(apply_chat_template)

# Train/val split â€” CRITICAL for evaluating overfitting
split = sft_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data = split["test"]

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print("\nFormatted training example:")
print(train_data[0]["text"][:500])

Map:   0%|          | 0/1032 [00:00<?, ? examples/s]

Train: 928 | Val: 104

Formatted training example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 27 Mar 2026

You are a knowledgeable medical AI assistant specializing in biomedical research. Provide accurate, evidence-based answers.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: Adhesive capsulitis is often difficult to diagnose in its early stage and to differentiate from other common shoulder disorders. The aim of this study was to validate any or all of the 8 clini


In [10]:
# Cell 7 - Configure and run SFT with T4-safe settings
import gc
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Initialize WandB run - this is your experiment tracker
wandb.init(
    project="medical-llm-finetune",
    name="llama-3.2-3b-medical-sft-t4-safe",
    config={
        "model": MODEL_NAME,
        "dataset": "medical-qa-shared-task-v1-toy+PubMedQA",
        "lora_r": 32,
        "lora_alpha": 64,
        "max_seq_length": MAX_SEQ_LEN,
        "training_samples": len(train_data),
    }
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,   # effective batch = 8
        warmup_steps=5,
        average_tokens_across_devices=False,
        num_train_epochs=3,
        learning_rate=2e-4,
        neftune_noise_alpha=5.0,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        optim="adamw_torch",             # FIX: Use native PyTorch optimizer to bypass the scaling bug
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        train_sampling_strategy="group_by_length",
        seed=42,
        output_dir="./sft_output",
        report_to="wandb",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        max_grad_norm=0.0,
    ),
)

gc.collect()
torch.cuda.empty_cache()

print("Starting SFT training...")
sft_stats = sft_trainer.train()
print(f"\nSFT Complete! Final train loss: {sft_stats.training_loss:.4f}")

wandb.finish()

Starting SFT training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 928 | Num Epochs = 3 | Total steps = 174
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


Step,Training Loss,Validation Loss
50,1.612909,1.596830
100,1.345950,1.624985
150,1.143124,1.676549


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transform


SFT Complete! Final train loss: 1.4070


eval/loss,▁▃█
eval/runtime,█▁▃
eval/samples_per_second,▁█▆
eval/steps_per_second,▁█▆
train/epoch,▁▁▂▂▃▃▃▄▄▄▅▅▅▆▆▇▇▇▇██
train/global_step,▁▁▂▂▃▃▃▄▄▄▅▅▅▆▆▇▇▇▇██
train/grad_norm,▇▄▅▅▂▁▄▃▄▅▄▄▇▆▅▅█
train/learning_rate,███▇▇▆▆▅▅▄▃▃▂▂▁▁▁
train/loss,█▅▅▅▅▄▃▃▃▃▃▃▁▂▂▁▂
eval/loss,1.67655
eval/runtime,15.7417


In [11]:

# Cell 8 - Evaluate with ROUGE and perplexity
import math
import warnings
import subprocess
import sys

try:
    import evaluate
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "evaluate", "rouge_score"])
    import evaluate

from tqdm import tqdm


warnings.filterwarnings(
    "ignore",
    message="The attention mask API under `transformers.modeling_attn_mask_utils`",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message="Both `max_new_tokens`",
    category=UserWarning,
)

rouge = evaluate.load("rouge")


def generate_response(messages, max_new_tokens=150):
    """Generate a deterministic response for evaluation."""
    FastLanguageModel.for_inference(model)
    model.eval()

    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    if isinstance(tokenized, torch.Tensor):
        input_ids = tokenized.to("cuda")
        attention_mask = None
    else:
        tokenized = tokenized.to("cuda")
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized.get("attention_mask")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = input_ids.shape[-1]
    new_tokens = outputs[0][prompt_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def compute_perplexity(dataset, batch_size=1):
    """Compute perplexity manually to avoid Trainer callback issues."""
    FastLanguageModel.for_inference(model)
    model.eval()

    losses = []
    for start_idx in tqdm(range(0, len(dataset), batch_size), desc="Perplexity"):
        batch = dataset[start_idx:start_idx + batch_size]["text"]
        encodings = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to("cuda")

        labels = encodings["input_ids"].clone()
        labels[encodings["attention_mask"] == 0] = -100

        with torch.no_grad():
            outputs = model(**encodings, labels=labels)

        losses.append(outputs.loss.item())

    mean_loss = sum(losses) / len(losses)
    return mean_loss, math.exp(mean_loss)


# Manual perplexity from the validation set
val_loss, perplexity = compute_perplexity(val_data)

# Run ROUGE on a small validation sample
sample_size = min(50, len(val_data))
eval_sample = val_data.select(range(sample_size))
predictions = []
references = []

print("Running ROUGE evaluation...")
for example in tqdm(eval_sample, desc="ROUGE"):
    messages = example["messages"]
    ref_answer = messages[2]["content"]

    prompt_msgs = [messages[0], messages[1]]
    pred = generate_response(prompt_msgs, max_new_tokens=150)
    predictions.append(pred)
    references.append(ref_answer)

results = rouge.compute(predictions=predictions, references=references)
print("\n=== EVALUATION RESULTS ===")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")
print(f"\nSample prediction:\n{predictions[0][:300]}")
print(f"\nSample reference:\n{references[0][:300]}")

Perplexity: 100%|██████████| 104/104 [00:21<00:00,  4.86it/s]


Running ROUGE evaluation...


ROUGE: 100%|██████████| 50/50 [04:22<00:00,  5.24s/it]


=== EVALUATION RESULTS ===
Validation Loss: 1.6811
Perplexity: 5.3717
ROUGE-1: 0.3641
ROUGE-2: 0.1371
ROUGE-L: 0.2813

Sample prediction:
The study findings suggest that the specialty of a physician may influence the frequency and depth of medication history documented in patients' medical records.

Sample reference:
Physicians appear to document more frequently and in greater depth medication history information that may aid the diagnostic tasks in their specific specialty. Researchers and other users of medication history data documented in patients' medical records by physicians may want to take special cogni


In [13]:
# Cell 9 - Build a stronger DPO preference dataset
import random
import re


dpo_source = train_data.select(range(min(500, len(train_data))))
answer_pool = [
    example["messages"][2]["content"].strip()
    for example in dpo_source
    if example["messages"][2]["content"].strip()
]


def sample_mismatched_answer(chosen_response):
    candidates = [answer for answer in answer_pool if answer != chosen_response]
    return random.choice(candidates) if candidates else "Please consult a healthcare professional for personalized advice."


def make_incomplete_answer(chosen_response):
    sentences = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", chosen_response.strip()) if sentence.strip()]
    if len(sentences) >= 2:
        return sentences[0] + " Please discuss the remaining details with a clinician."

    words = chosen_response.split()
    shortened = " ".join(words[: min(20, len(words))]).strip()
    if shortened and shortened[-1] not in ".!?":
        shortened += "."
    return shortened + " Please discuss the remaining details with a clinician."


def make_overcautious_answer(chosen_response):
    return "This depends heavily on the patient and clinical context. A clinician should review the case directly before any conclusion is made."


def make_binary_flip_answer(chosen_response):
    stripped = chosen_response.strip()
    lowered = stripped.lower()
    if lowered.startswith("yes"):
        return re.sub(r"^[Yy]es", "No", stripped, count=1)
    if lowered.startswith("no"):
        return re.sub(r"^[Nn]o", "Yes", stripped, count=1)
    return None


def create_dpo_pair(example):
    """Create harder prompt / chosen / rejected triples for DPO training."""
    messages = example["messages"]
    prompt = tokenizer.apply_chat_template(
        [messages[0], messages[1]],
        tokenize=False,
        add_generation_prompt=True,
    )
    chosen_response = messages[2]["content"].strip()

    strategies = [
        ("mismatched_answer", lambda: sample_mismatched_answer(chosen_response)),
        ("incomplete_answer", lambda: make_incomplete_answer(chosen_response)),
        ("overcautious_answer", lambda: make_overcautious_answer(chosen_response)),
    ]

    flipped_answer = make_binary_flip_answer(chosen_response)
    if flipped_answer is not None and flipped_answer != chosen_response:
        strategies.append(("binary_flip", lambda: flipped_answer))

    strategy_name, strategy_fn = random.choice(strategies)
    rejected_response = strategy_fn().strip()
    if rejected_response == chosen_response:
        strategy_name = "mismatched_answer"
        rejected_response = sample_mismatched_answer(chosen_response)

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
        "rejection_strategy": strategy_name,
    }


dpo_dataset = dpo_source.map(
    create_dpo_pair,
    remove_columns=dpo_source.column_names,
)

print(f"DPO dataset: {len(dpo_dataset)} preference pairs")
print("\nRejection strategy counts:")
strategy_counts = {}
for row in dpo_dataset:
    strategy_counts[row["rejection_strategy"]] = strategy_counts.get(row["rejection_strategy"], 0) + 1
print(strategy_counts)

print("\nSample DPO pair:")
print(f"Prompt: {dpo_dataset[0]['prompt'][:120]}...")
print(f"Chosen: {dpo_dataset[0]['chosen'][:120]}...")
print(f"Rejected ({dpo_dataset[0]['rejection_strategy']}): {dpo_dataset[0]['rejected'][:160]}...")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DPO dataset: 500 preference pairs

Rejection strategy counts:
{'mismatched_answer': 178, 'overcautious_answer': 160, 'incomplete_answer': 159, 'binary_flip': 3}

Sample DPO pair:
Prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 27 Mar 20...
Chosen: None of the clinical identifiers for early-stage adhesive capsulitis previously proposed by expert consensus have been v...
Rejected (mismatched_answer): These data suggest that being willing and fit enough for a chemotherapy protocol is a good prognostic factor for invasive bladder cancer. This eligibility bias ...


In [14]:
# Cell 10 - Run DPO training without TRL dependency issues
import gc
import random
import torch
import torch.nn.functional as F
from tqdm import tqdm

BETA = 0.2
DPO_LEARNING_RATE = 5e-6
DPO_GRAD_ACCUM = 8
DPO_EPOCHS = 1
REFERENCE_ADAPTER_NAME = "reference"

if wandb.run is not None:
    wandb.finish()

model.save_pretrained("./sft_adapter_checkpoint")
tokenizer.save_pretrained("./sft_adapter_checkpoint")
tokenizer.pad_token = tokenizer.eos_token

if not hasattr(model, "load_adapter") or not hasattr(model, "set_adapter"):
    raise RuntimeError("This DPO fallback expects a PEFT model with load_adapter() and set_adapter().")


def get_active_adapter_name(peft_model):
    active_adapter = getattr(peft_model, "active_adapter", None)
    if isinstance(active_adapter, str) and active_adapter:
        return active_adapter

    active_adapters = getattr(peft_model, "active_adapters", None)
    if isinstance(active_adapters, str) and active_adapters:
        return active_adapters
    if isinstance(active_adapters, (list, tuple)) and len(active_adapters) > 0:
        return active_adapters[0]

    return "default"


def set_adapter_safe(adapter_name, inference_mode=False):
    try:
        model.set_adapter(adapter_name, inference_mode=inference_mode)
    except TypeError:
        model.set_adapter(adapter_name)
        if hasattr(model, "set_requires_grad"):
            model.set_requires_grad(adapter_name, not inference_mode)


TRAIN_ADAPTER_NAME = get_active_adapter_name(model)
EXISTING_ADAPTERS = list(getattr(model, "peft_config", {}).keys()) if isinstance(getattr(model, "peft_config", None), dict) else []
if REFERENCE_ADAPTER_NAME not in EXISTING_ADAPTERS:
    model.load_adapter(
        "./sft_adapter_checkpoint",
        adapter_name=REFERENCE_ADAPTER_NAME,
        is_trainable=False,
    )


def build_inputs(prompt_text, response_text):
    full_text = prompt_text + response_text
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )["input_ids"]
    encodings = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to("cuda")

    labels = encodings["input_ids"].clone()
    prompt_length = min(len(prompt_ids), labels.shape[1] - 1)
    labels[:, :prompt_length] = -100
    return encodings, labels


def sequence_logprob_for_active_adapter(prompt_text, response_text):
    encodings, labels = build_inputs(prompt_text, response_text)
    outputs = model(**encodings)

    logits = outputs.logits[:, :-1, :]
    shifted_labels = labels[:, 1:]
    loss_mask = shifted_labels != -100

    safe_labels = shifted_labels.clone()
    safe_labels[~loss_mask] = 0

    token_logps = F.log_softmax(logits, dim=-1).gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
    sequence_logp = (token_logps * loss_mask).sum(dim=-1)
    return sequence_logp


# Convert to a mutable list so we can cache reference log-probs
preference_examples = [dpo_dataset[i] for i in range(len(dpo_dataset))]

print("Precomputing reference log-probs...")
set_adapter_safe(REFERENCE_ADAPTER_NAME, inference_mode=True)
model.eval()
for example in tqdm(preference_examples, desc="Reference"):
    with torch.no_grad():
        example["ref_chosen_logp"] = sequence_logprob_for_active_adapter(example["prompt"], example["chosen"]).item()
        example["ref_rejected_logp"] = sequence_logprob_for_active_adapter(example["prompt"], example["rejected"]).item()

gc.collect()
torch.cuda.empty_cache()

set_adapter_safe(TRAIN_ADAPTER_NAME, inference_mode=False)
if hasattr(model, "for_training"):
    try:
        model.for_training(use_gradient_checkpointing=True)
    except TypeError:
        model.for_training()
model.train()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

trainable_params = [p for p in model.parameters() if p.requires_grad]
try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.AdamW8bit(trainable_params, lr=DPO_LEARNING_RATE)
    print("Using bitsandbytes AdamW8bit optimizer")
except Exception:
    optimizer = torch.optim.AdamW(trainable_params, lr=DPO_LEARNING_RATE)
    print("Using torch.optim.AdamW optimizer")

wandb.init(
    project="medical-llm-finetune",
    name="llama-3.2-3b-medical-dpo-custom",
    config={
        "phase": "DPO-custom",
        "beta": BETA,
        "dpo_pairs": len(preference_examples),
        "max_seq_length": MAX_SEQ_LEN,
        "learning_rate": DPO_LEARNING_RATE,
        "gradient_accumulation_steps": DPO_GRAD_ACCUM,
    },
)

optimizer.zero_grad()
global_step = 0
running_loss = 0.0

print("Starting custom DPO training...")
for epoch in range(DPO_EPOCHS):
    random.shuffle(preference_examples)
    progress_bar = tqdm(range(len(preference_examples)), desc=f"DPO Epoch {epoch + 1}/{DPO_EPOCHS}")

    for idx in progress_bar:
        example = preference_examples[idx]
        set_adapter_safe(TRAIN_ADAPTER_NAME, inference_mode=False)

        chosen_logp = sequence_logprob_for_active_adapter(example["prompt"], example["chosen"])
        rejected_logp = sequence_logprob_for_active_adapter(example["prompt"], example["rejected"])

        ref_chosen = torch.tensor(example["ref_chosen_logp"], device=chosen_logp.device)
        ref_rejected = torch.tensor(example["ref_rejected_logp"], device=chosen_logp.device)

        preference_logit = BETA * ((chosen_logp - rejected_logp) - (ref_chosen - ref_rejected))
        loss = -F.logsigmoid(preference_logit).mean()

        (loss / DPO_GRAD_ACCUM).backward()
        running_loss += loss.item()

        should_step = ((idx + 1) % DPO_GRAD_ACCUM == 0) or (idx == len(preference_examples) - 1)
        if should_step:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

            average_loss = running_loss / (idx + 1)
            progress_bar.set_postfix(loss=f"{average_loss:.4f}")
            if global_step % 10 == 0:
                wandb.log({
                    "dpo/loss": loss.item(),
                    "dpo/avg_loss": average_loss,
                    "dpo/global_step": global_step,
                    "epoch": epoch + 1,
                })

gc.collect()
torch.cuda.empty_cache()
set_adapter_safe(TRAIN_ADAPTER_NAME, inference_mode=False)
model.save_pretrained("./dpo_output")
tokenizer.save_pretrained("./dpo_output")

final_loss = running_loss / max(len(preference_examples), 1)
print(f"Custom DPO training complete! Final average loss: {final_loss:.4f}")
wandb.finish()

Precomputing reference log-probs...


Reference: 100%|██████████| 500/500 [03:26<00:00,  2.42it/s]


Using bitsandbytes AdamW8bit optimizer


Starting custom DPO training...


DPO Epoch 1/1:   1%|          | 6/500 [00:06<09:30,  1.15s/it]

Unsloth: Will smartly offload gradients to save VRAM!


DPO Epoch 1/1: 100%|██████████| 500/500 [09:57<00:00,  1.19s/it, loss=0.0979]


Custom DPO training complete! Final average loss: 0.0979


dpo/avg_loss,█▅▃▂▁▁
dpo/global_step,▁▂▄▅▇█
dpo/loss,█▃▁▁▁▁
epoch,▁▁▁▁▁▁
dpo/avg_loss,0.10156
dpo/global_step,60
dpo/loss,0.00053
epoch,1
